<a href="https://colab.research.google.com/github/bonsii2/DiseasePredictoreFromSysmptom/blob/model/DiseasePrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Libraries & Load Dataset

In [51]:
import pandas as pd
import numpy as np

In [53]:
data = pd.read_csv('medical_dataset_10000.csv')

In [54]:
data.head()

,Age,Gender,Symptoms,Symptom_Count,Disease
0,22,Male,"headache, chills",2,Malaria
1,11,Female,"chest pain, fatigue, dizziness, chills",4,Heart Disease
2,40,Female,"runny nose, rash, itchy eyes",3,Allergy
3,24,Female,"fever, chest pain, shortness of breath",3,Pneumonia
4,86,Male,"nausea, fever, sweating",3,Malaria


# Check Dataset Shape and Info

In [55]:
print('shape of the dataset', data.shape)

shape of the dataset (10000, 5)


In [56]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Age            10000 non-null  int64 
 1   Gender         10000 non-null  object
 2   Symptoms       10000 non-null  object
 3   Symptom_Count  10000 non-null  int64 
 4   Disease        10000 non-null  object
dtypes: int64(2), object(3)
memory usage: 390.8+ KB


# Check Missing Values

In [57]:
data.isnull().sum()

,0
Age,0
Gender,0
Symptoms,0
Symptom_Count,0
Disease,0


# Drop Unnecessary Column

In [59]:
data.head()

,Age,Gender,Symptoms,Symptom_Count,Disease
0,22,Male,"headache, chills",2,Malaria
1,11,Female,"chest pain, fatigue, dizziness, chills",4,Heart Disease
2,40,Female,"runny nose, rash, itchy eyes",3,Allergy
3,24,Female,"fever, chest pain, shortness of breath",3,Pneumonia
4,86,Male,"nausea, fever, sweating",3,Malaria


In [60]:
data['Disease'].value_counts()

,count
Disease,
Tuberculosis,1058
Heart Disease,1016
Diabetes,1010
Allergy,1008
Pneumonia,1003
Malaria,1000
Flu,999
Stroke,973
Thyroid Disorder,973


# Encode Gender Column

---
ML models cannot understand text like "Male", "Female", "Other".


In [61]:
from sklearn.preprocessing import LabelEncoder

le_gender = LabelEncoder()
data['Gender'] = le_gender.fit_transform(data['Gender'])

data.head()


,Age,Gender,Symptoms,Symptom_Count,Disease
0,22,1,"headache, chills",2,Malaria
1,11,0,"chest pain, fatigue, dizziness, chills",4,Heart Disease
2,40,0,"runny nose, rash, itchy eyes",3,Allergy
3,24,0,"fever, chest pain, shortness of breath",3,Pneumonia
4,86,1,"nausea, fever, sweating",3,Malaria


# Convert Symptoms Text into Numerical Features (TF-IDF)

In [12]:
# from sklearn.feature_extraction.text import TfidfVectorizer

# tfidf = TfidfVectorizer(max_features=3000)
# x_symptoms = tfidf.fit_transform(data['Symptoms'])

# Extract All Unique Symptoms

In [62]:
from sklearn.preprocessing import MultiLabelBinarizer

# Split symptoms into list
data["Symptoms_List"] = data["Symptoms"].apply(lambda x: [s.strip() for s in x.split(",")])

# Create binary symptom matrix
mlb = MultiLabelBinarizer()
X_symptoms = mlb.fit_transform(data["Symptoms_List"])

print("Number of unique symptoms:", len(mlb.classes_))


Number of unique symptoms: 28


# Combine All Features

In [63]:
from scipy.sparse import hstack

x_other = data[['Age', 'Gender', 'Symptom_Count']].values
x = np.hstack([x_other, X_symptoms])

# Encode Target (Disease)

In [64]:
le_desease = LabelEncoder()
y = le_desease.fit_transform(data['Disease'])

# Train Test Split

In [65]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.2, random_state = 42, stratify = y)

# Train Random Forest Model

In [66]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
    )
model.fit(x_train, y_train)

RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)

# Model Evaluation

In [67]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(x_test)

print('Accuracy', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy 0.977
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       202
           1       0.99      0.97      0.98       202
           2       0.97      0.95      0.96       200
           3       0.95      0.96      0.95       203
           4       0.97      0.98      0.98       192
           5       0.99      1.00      1.00       200
           6       0.93      0.94      0.94       200
           7       1.00      1.00      1.00       195
           8       0.97      0.99      0.98       195
           9       1.00      0.98      0.99       211

    accuracy                           0.98      2000
   macro avg       0.98      0.98      0.98      2000
weighted avg       0.98      0.98      0.98      2000



## Save the Model and Tools

In [68]:
import joblib

joblib.dump(model, 'model.pkl')
joblib.dump(tfidf, 'tfidf.pkl')
joblib.dump(le_gender, 'le_gender.pkl')
joblib.dump(le_desease, 'le_desease.pkl')


['le_desease.pkl']